# Closing three reviewer gaps

Three objections currently sit in the Limitations section as disclosures. Each is
answerable with the pipeline that already exists, and answering them converts a
concession into a result.

**Gap 1 — window overlap.** Windows are 120 beats at step 5, so adjacent windows
share ~96% of their samples. The 11,846 windows are far from independent. LOSO
prevents subject leakage but does nothing about this, and it plausibly flatters
exactly the features the paper credits: local-deviation features benefit most from
near-duplicate neighbours. **Fix:** rerun the decomposition at steps 5, 20, 40, 60
and check whether the residual-versus-level pattern survives at low overlap.

**Gap 2 — the timescale sweep is partly foregone.** Every period tested (3–24 h)
exceeds the 1.5 h mean recording span, so over that span they all approach a
near-linear trend. Finding them indistinguishable follows partly from the design.
**Fix:** extend the sweep to periods *shorter* than the recording — 15 min, 30 min,
1 h — where the fitted forms genuinely diverge.

**Gap 3 — Table I has no dispersion.** Section IV-B explains the 0.646 to 0.642
drop as fold variation, but the table reports no standard deviations or intervals,
so the explanation is asserted rather than supported. **Fix:** report per-fold
standard deviations and subject-level bootstrap intervals.

---

**Prerequisites.** Run in the same session as `notebook-rigorous-analysis.ipynb`
after A1–A5 (config, helpers, preprocessing, model helper). Reuses `wesad`,
`wesad_cos`, `hrv_features`, `resid_features`, `circ_features`, `fit_cos`,
`cos_model` and `xgb_loso_predictions`. U0–U5 are not needed.

**Runtime.** Roughly 35–45 minutes: gap 1 is 16 LOSO runs on progressively smaller
matrices, gap 2 is 6 runs plus refitting, gap 3 is cheap. Each section saves its
results to `/kaggle/working` as it finishes, so a crash costs one section, not all
three.

In [1]:
!pip install neurokit2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 11.9 MB/s eta 0:00:00


In [2]:
import os, json, pickle, warnings, itertools
import numpy as np, pandas as pd
from scipy.signal import welch
from scipy.optimize import curve_fit
from scipy import stats
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, mean_absolute_error)
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
np.random.seed(42)
print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU'))>0)

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE_PATH='/kaggle/working/processed'; os.makedirs(SAVE_PATH, exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']
N_BOOT=10000
RNG=np.random.RandomState(42)

FEATURE_NAMES=['mean_rr','sdnn','rmssd','pnn50','cv_rr','vlf','lf','hf',
               'lf_hf','lf_nu','sd1','sd2','sd_ratio']
RESIDUAL_NAMES=['res_mean','res_std','res_max','res_trend','res_energy']
CIRC_NAMES=['sin_24h','cos_24h','sin_90m','cos_90m','cortisol']
print("Config loaded.")

TF 2.20.0 GPU True
Config loaded.


In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg=nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _,info=nk.ecg_peaks(ecg, sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap=np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]
        out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)

def hrv_features(w, fs=4.0):
    rr,diff=np.array(w),np.diff(w)
    mean_rr=np.mean(rr); sdnn=np.std(rr); rmssd=np.sqrt(np.mean(diff**2))
    pnn50=np.sum(np.abs(diff)>50)/len(diff)*100; cv=sdnn/mean_rr
    t=np.cumsum(rr)/1000.0; u=np.interp(np.arange(0,t[-1],1/fs),t,rr)
    fr,psd=welch(u,fs=fs,nperseg=min(256,len(u)))
    vlf=TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf=TRAPZ(psd[(fr>=0.04)&(fr<0.15)]); hf=TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf=lf/(hf+1e-8); lf_nu=lf/(lf+hf+1e-8)
    sd1=np.sqrt(0.5)*np.std(diff); sd2=np.sqrt(max(2*sdnn**2-0.5*np.var(diff),0)); sdr=sd1/(sd2+1e-8)
    return np.array([mean_rr,sdnn,rmssd,pnn50,cv,vlf,lf,hf,lf_hf,lf_nu,sd1,sd2,sdr])

def resid_features(rw):
    r=np.array(rw)
    return np.array([np.mean(r),np.std(r),np.max(np.abs(r)),np.polyfit(np.arange(len(r)),r,1)[0],np.sum(r**2)/len(r)])

def circ_features(ts):
    t,hour=ts%86400,(ts%86400)/3600.0
    cort=0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),cort])

def cos_model(th,m,a,p): return m+a*np.cos((2*np.pi/24.0)*th+p)
def fit_cos(sig,ts,p0):
    th=(ts%86400)/3600.0
    try:
        popt,_=curve_fit(cos_model,th,sig,p0=p0,maxfev=10000)
        base=cos_model(th,*popt)
        r2=1-np.sum((sig-base)**2)/np.sum((sig-np.mean(sig))**2)
        return base,sig-base,popt[0],popt[1],r2
    except RuntimeError:
        return np.full_like(sig,np.mean(sig)),sig-np.mean(sig),float(np.mean(sig)),0.0,0.0

def roll_rmssd(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; d=np.diff(w); o[i]=np.sqrt(np.mean(d**2)) if len(d)>1 else 0
    return o
def roll_sdnn(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; o[i]=np.std(w) if len(w)>1 else 0
    return o
print("Helpers defined.")

Helpers defined.


In [4]:
wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr_from_ecg(ecg); temp=align_temp(wt,rp)
        rr,ts=clean_rr(rr,ts); rl=labels_to_rr(labels,rp)
        keep=rl>0; rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w); loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']={'rr_ms':rrk,'temp':tk,'timestamps':tsk,'labels':new}
    except Exception as e: print(f"S{sid} FAIL {e}")
print(f"{len(wesad)} subjects")

wesad_cos={}
for sid,d in wesad.items():
    _,rr_res,mesor,amp,r2h=fit_cos(d['rr_ms'],d['timestamps'],[np.mean(d['rr_ms']),50.0,-1.5])
    _,temp_res,tm,ta,r2t=fit_cos(d['temp'],d['timestamps'],[np.mean(d['temp']),1.0,-1.5])
    wesad_cos[sid]={'rr_res':rr_res,'temp_res':temp_res,'mesor':mesor,'amplitude':amp,
                    'r2_hrv':r2h,'r2_temp':r2t,'temp_mesor':tm,'temp_amp':ta}
print("Cosinor fitted.")

15 subjects
Cosinor fitted.


In [5]:
def build_all(data, cos, window=120, step=5):
    Xseq,Xcirc,Xxgb,y,g,hourv,phasev=[],[],[],[],[],[],[]
    # decomposition columns kept separately for U1
    mesor_col, resid_cols, time_cols = [], [], []
    for sid,d in data.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        rr_res,temp_res=cos[sid]['rr_res'],cos[sid]['temp_res']
        mesor,amp=cos[sid]['mesor'],cos[sid]['amplitude']
        rn=(rr-np.mean(rr))/(np.std(rr)+1e-8)
        tn=(temp-np.mean(temp))/(np.std(temp)+1e-8)
        rrn=(rr_res-np.mean(rr_res))/(np.std(rr_res)+1e-8)
        trn=(temp_res-np.mean(temp_res))/(np.std(temp_res)+1e-8)
        rm,sd=roll_rmssd(rn),roll_sdnn(rn); hr=60000/(rr+1e-8)
        t0,t1=ts.min(),ts.max(); dur=max(t1-t0,1e-6)
        for s in range(0,len(rn)-window,step):
            e=s+window; lab=labels[s+window//2]; bi=min(s+window//2,len(ts)-1)
            seq=np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1)
            t=ts[bi]%86400; hour=t/3600.0
            circ=np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
                0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
            try:
                hf=hrv_features(rr[s:e]); rf=resid_features(rr_res[s:e]); cf=circ_features(ts[bi])
                xgbf=np.concatenate([hf,rf,np.array([mesor,amp]),cf])
            except Exception: continue
            Xseq.append(seq);Xcirc.append(circ);Xxgb.append(xgbf)
            y.append(lab);g.append(int(sid[1:]))
            hourv.append(ts[bi]-t0)                 # seconds since session start
            phasev.append((ts[bi]-t0)/dur)          # normalised phase 0..1
            mesor_col.append([mesor,amp])
            resid_cols.append(rf)
            time_cols.append(cf)
    return (np.array(Xseq,dtype=np.float32),np.array(Xcirc,dtype=np.float32),
            np.array(Xxgb),np.array(y,dtype=np.int32),np.array(g,dtype=np.int32),
            np.array(hourv),np.array(phasev),np.array(mesor_col),
            np.array(resid_cols),np.array(time_cols))

(X_seq,X_circ,X_xgb,y_all,groups,sess_sec,phase,
 MESOR_COL,RESID_COL,TIME_COL)=build_all(wesad,wesad_cos)
print("seq",X_seq.shape,"xgb",X_xgb.shape,"classes",np.bincount(y_all))
print("phase range",phase.min(),phase.max())

# HRV-only feature block = first 13 columns + circadian last 5 (no residual/mesor)
HRV_IDX=list(range(0,13))+list(range(20,25))    # 13 hrv + 5 circadian
print("HRV-only feature dim:",len(HRV_IDX),"| full dim:",X_xgb.shape[1])

seq (11846, 120, 7) xgb (11846, 25) classes [8594 1101 1080 1071]
phase range 0.0077978789368999 0.9915185703251734
HRV-only feature dim: 18 | full dim: 25


In [6]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_cnn(window=120,nch=7,ncirc=7,ncls=4):
    si=tf.keras.Input(shape=(window,nch),name='sequence')
    ci=tf.keras.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x]); x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    out=layers.Dense(ncls,activation='softmax')(x)
    return Model([si,ci],out,name='CNN_BiLSTM_Attn_v2')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=42,n_jobs=-1)

def xgb_loso_predictions(X, y, groups, cols=None):
    """Return pooled y_true, y_pred, y_proba, subject_ids in fold order."""
    logo=LeaveOneGroupOut()
    yt,yp,pp,subj=[],[],[],[]
    perfold={}
    for tr,te in logo.split(X,y,groups):
        s=int(np.unique(groups[te])[0])
        Xtr=X[tr][:,cols] if cols is not None else X[tr]
        Xte=X[te][:,cols] if cols is not None else X[te]
        sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
        m=make_xgb(); m.fit(Xtr,y[tr],sample_weight=compute_sample_weight('balanced',y[tr]),verbose=False)
        proba=m.predict_proba(Xte); pred=np.argmax(proba,axis=1)
        yt.extend(y[te]); yp.extend(pred); pp.extend(proba); subj.extend([s]*len(te))
        perfold[s]=f1_score(y[te],pred,average='macro',zero_division=0)
    return (np.array(yt),np.array(yp),np.array(pp),np.array(subj),perfold)
print("Models + LOSO helper defined.")

Models + LOSO helper defined.


## 0. Dependency check

In [7]:
import os, json, time
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score

_needed = ['wesad', 'wesad_cos', 'hrv_features', 'resid_features', 'circ_features',
           'fit_cos', 'cos_model', 'xgb_loso_predictions']
_missing = [n for n in _needed if n not in dir()]
assert not _missing, (f"missing from session: {_missing}. "
                      "Run notebook-rigorous-analysis.ipynb A1-A5 first.")
assert len(wesad) == 15, f"expected 15 subjects, found {len(wesad)}"

SAVE = '/kaggle/working'
C_HRV, C_RESID, C_LEVEL, C_CIRC = list(range(13)), list(range(13,18)), [18,19], list(range(20,25))
CONFIGS = {
    'baseline (HRV+time)'  : C_HRV + C_CIRC,
    '+ level'              : C_HRV + C_CIRC + C_LEVEL,
    '+ residual'           : C_HRV + C_CIRC + C_RESID,
    '+ level + residual'   : C_HRV + C_CIRC + C_LEVEL + C_RESID,
}
print("dependency check passed")

dependency check passed


## Gap 1 — step-size sensitivity

The feature construction is unchanged; only the stride between window starts moves.
Larger steps mean fewer, less correlated windows.

| step | overlap | approx. windows |
| --- | --- | --- |
| 5 | 95.8% | ~11,850 |
| 20 | 83.3% | ~2,960 |
| 40 | 66.7% | ~1,480 |
| 60 | 50.0% | ~990 |

The level term uses the **session mean and SD**, not the degenerate MESOR — the
baseline-control notebook established that the fitted MESOR is uninformative for
trivial reasons, so testing it again here would answer nothing.

In [8]:
WINDOW = 120

def build_at_step(step, level='session'):
    X, y, g = [], [], []
    for sid, d in wesad.items():
        rr, ts, lab = d['rr_ms'], d['timestamps'], d['labels']
        rr_res = wesad_cos[sid]['rr_res']
        if level == 'session':
            lv = np.array([float(np.mean(rr)), float(np.std(rr))])
        else:
            lv = np.array([wesad_cos[sid]['mesor'], wesad_cos[sid]['amplitude']])
        for s in range(0, len(rr)-WINDOW, step):
            e = s + WINDOW; mid = s + WINDOW//2; bi = min(mid, len(ts)-1)
            try:
                f = np.concatenate([hrv_features(rr[s:e]), resid_features(rr_res[s:e]),
                                    lv, circ_features(ts[bi])])
            except Exception:
                continue
            X.append(f); y.append(lab[mid]); g.append(int(sid[1:]))
    return np.array(X), np.array(y, int), np.array(g, int)

STEPS = [5, 20, 40, 60]
step_results = {}
t0 = time.time()
for st in STEPS:
    X, y, g = build_at_step(st)
    row = {'n_windows': int(len(y)), 'overlap_pct': round(100*(1 - st/WINDOW), 1)}
    print(f"\nstep {st:2d}  overlap {row['overlap_pct']:.1f}%  windows {len(y)}")
    for name, cols in CONFIGS.items():
        yt, yp, _, subj, perfold = xgb_loso_predictions(X, y, g, cols=cols)
        f1 = f1_score(yt, yp, average='macro', zero_division=0)
        k  = cohen_kappa_score(yt, yp, weights='quadratic')
        row[name] = {'f1': float(f1), 'kappa': float(k),
                     'perfold': {str(s): float(v) for s, v in perfold.items()}}
        print(f"   {name:<22} F1={f1:.3f}  kappa={k:.3f}")
    base = row['baseline (HRV+time)']['f1']
    row['d_level'] = row['+ level']['f1'] - base
    row['d_resid'] = row['+ residual']['f1'] - base
    row['d_both']  = row['+ level + residual']['f1'] - base
    print(f"   {'-> level':<22} {row['d_level']:+.3f}")
    print(f"   {'-> residual':<22} {row['d_resid']:+.3f}")
    step_results[st] = row

json.dump(step_results, open(f'{SAVE}/gap1_stepsize.json','w'))
print(f"\nelapsed {(time.time()-t0)/60:.1f} min -> saved gap1_stepsize.json")


step  5  overlap 95.8%  windows 11846
   baseline (HRV+time)    F1=0.551  kappa=0.668
   + level                F1=0.602  kappa=0.777
   + residual             F1=0.646  kappa=0.834
   + level + residual     F1=0.643  kappa=0.838
   -> level               +0.051
   -> residual            +0.096

step 20  overlap 83.3%  windows 2966
   baseline (HRV+time)    F1=0.531  kappa=0.659
   + level                F1=0.573  kappa=0.763
   + residual             F1=0.626  kappa=0.824
   + level + residual     F1=0.625  kappa=0.831
   -> level               +0.042
   -> residual            +0.095

step 40  overlap 66.7%  windows 1487
   baseline (HRV+time)    F1=0.504  kappa=0.648
   + level                F1=0.541  kappa=0.746
   + residual             F1=0.565  kappa=0.788
   + level + residual     F1=0.574  kappa=0.803
   -> level               +0.037
   -> residual            +0.061

step 60  overlap 50.0%  windows 993
   baseline (HRV+time)    F1=0.523  kappa=0.668
   + level                

In [9]:
print("="*80)
print("GAP 1 RESULT: does the mechanism finding survive reduced window overlap?")
print("="*80)
print(f"{'step':>5}{'overlap':>10}{'windows':>10}{'d_level':>12}{'d_resid':>12}{'ratio':>10}")
print("-"*80)
for st in STEPS:
    r = step_results[st]
    ratio = r['d_resid']/max(abs(r['d_level']), 1e-9)
    print(f"{st:>5}{r['overlap_pct']:>9.1f}%{r['n_windows']:>10}"
          f"{r['d_level']:>+12.3f}{r['d_resid']:>+12.3f}{ratio:>9.1f}x")
print("-"*80)

r5, r60 = step_results[5], step_results[60]
drift_resid = abs(r60['d_resid'] - r5['d_resid'])
sign_stable = all(step_results[s]['d_resid'] > step_results[s]['d_level'] for s in STEPS)
print(f"\nresidual contribution drift, step 5 -> 60: {drift_resid:.3f}")
print(f"residual exceeds level at every step:     {sign_stable}")
print()
if sign_stable and drift_resid < 0.03:
    print("OUTCOME: the finding is stable under reduced overlap. The dependence")
    print("between windows is not manufacturing the result. Report the table above")
    print("and replace the overlap paragraph in Limitations with this result.")
elif sign_stable:
    print(f"OUTCOME: the ordering holds at every step, but the magnitude moves by")
    print(f"{drift_resid:.3f} across the range. The qualitative claim survives; report")
    print("the drift honestly and keep a shortened caveat about effect size.")
else:
    print("OUTCOME: the ordering does NOT hold at every step. The finding is")
    print("partly an artefact of window overlap. This must be addressed before")
    print("submission, not disclosed in Limitations.")

GAP 1 RESULT: does the mechanism finding survive reduced window overlap?
 step   overlap   windows     d_level     d_resid     ratio
--------------------------------------------------------------------------------
    5     95.8%     11846      +0.051      +0.096      1.9x
   20     83.3%      2966      +0.042      +0.095      2.3x
   40     66.7%      1487      +0.037      +0.061      1.6x
   60     50.0%       993      +0.033      +0.061      1.8x
--------------------------------------------------------------------------------

residual contribution drift, step 5 -> 60: 0.035
residual exceeds level at every step:     True

OUTCOME: the ordering holds at every step, but the magnitude moves by
0.035 across the range. The qualitative claim survives; report
the drift honestly and keep a shortened caveat about effect size.


## Gap 2 — timescales shorter than the recording

Periods of 3–24 h over a 1.5 h window all approach a near-linear trend, so their
similarity is partly structural. Periods of 15–60 min complete one or more cycles
within the recording and therefore fit genuinely different shapes.

`fit_cos` hardcodes a 24-hour period, so this section uses a period-parameterised
version. Everything else — window size, features, folds — is unchanged, and the
24 h condition is included so the new numbers can be checked against Table II.

In [10]:
def cos_model_p(th, m, a, p, period_h):
    return m + a*np.cos((2*np.pi/period_h)*th + p)

def fit_cos_period(sig, ts, period_h):
    from scipy.optimize import curve_fit
    th = (ts % 86400)/3600.0
    f = lambda t, m, a, p: cos_model_p(t, m, a, p, period_h)
    try:
        popt, _ = curve_fit(f, th, sig, p0=[np.mean(sig), 50.0, -1.5], maxfev=10000)
        base = f(th, *popt)
        return sig - base, float(popt[0]), float(abs(popt[1]))
    except RuntimeError:
        return sig - np.mean(sig), float(np.mean(sig)), 0.0

PERIODS = [0.25, 0.5, 1.0, 3.0, 6.0, 12.0, 24.0]   # hours
STEP_G2 = 5

def build_period(period_h):
    X, y, g = [], [], []
    for sid, d in wesad.items():
        rr, ts, lab = d['rr_ms'], d['timestamps'], d['labels']
        res, mesor, amp = fit_cos_period(rr, ts, period_h)
        for s in range(0, len(rr)-WINDOW, STEP_G2):
            e = s + WINDOW; mid = s + WINDOW//2; bi = min(mid, len(ts)-1)
            try:
                f = np.concatenate([hrv_features(rr[s:e]), resid_features(res[s:e]),
                                    np.array([mesor, amp]), circ_features(ts[bi])])
            except Exception:
                continue
            X.append(f); y.append(lab[mid]); g.append(int(sid[1:]))
    return np.array(X), np.array(y, int), np.array(g, int)

FULL = C_HRV + C_CIRC + C_LEVEL + C_RESID
period_results = {}
t0 = time.time()
for ph in PERIODS:
    X, y, g = build_period(ph)
    yt, yp, _, _, perfold = xgb_loso_predictions(X, y, g, cols=FULL)
    f1 = f1_score(yt, yp, average='macro', zero_division=0)
    k  = cohen_kappa_score(yt, yp, weights='quadratic')
    period_results[ph] = {'f1': float(f1), 'kappa': float(k),
                          'perfold': {str(s): float(v) for s, v in perfold.items()}}
    lbl = f"{int(ph*60)} min" if ph < 1 else f"{ph:.0f} h"
    mark = "  <- shorter than recording" if ph <= 1.0 else ""
    print(f"period {lbl:>7}   F1={f1:.3f}  kappa={k:.3f}{mark}")

json.dump({str(k_): v for k_, v in period_results.items()},
          open(f'{SAVE}/gap2_periods.json','w'))
print(f"\nelapsed {(time.time()-t0)/60:.1f} min -> saved gap2_periods.json")

period  15 min   F1=0.587  kappa=0.827  <- shorter than recording
period  30 min   F1=0.583  kappa=0.809  <- shorter than recording
period     1 h   F1=0.576  kappa=0.773  <- shorter than recording
period     3 h   F1=0.640  kappa=0.840
period     6 h   F1=0.647  kappa=0.845
period    12 h   F1=0.639  kappa=0.834
period    24 h   F1=0.645  kappa=0.838

elapsed 6.2 min -> saved gap2_periods.json


In [11]:
from scipy.stats import wilcoxon
print("="*80)
print("GAP 2 RESULT: do sub-recording timescales behave differently?")
print("="*80)
short = [p for p in PERIODS if p <= 1.0]
long_ = [p for p in PERIODS if p > 1.0]
f1_short = [period_results[p]['f1'] for p in short]
f1_long  = [period_results[p]['f1'] for p in long_]
print(f"sub-recording  ({len(short)} periods): F1 {np.mean(f1_short):.3f} "
      f"(range {min(f1_short):.3f}-{max(f1_short):.3f})")
print(f"supra-recording({len(long_)} periods): F1 {np.mean(f1_long):.3f} "
      f"(range {min(f1_long):.3f}-{max(f1_long):.3f})")
print(f"spread across ALL periods: {max(period_results[p]['f1'] for p in PERIODS) - min(period_results[p]['f1'] for p in PERIODS):.3f}")

# paired per-subject test: best short vs 24h
best_short = max(short, key=lambda p: period_results[p]['f1'])
a = np.array([period_results[best_short]['perfold'][k] for k in sorted(period_results[24.0]['perfold'])])
b = np.array([period_results[24.0]['perfold'][k]       for k in sorted(period_results[24.0]['perfold'])])
st, pv = wilcoxon(a, b)
lbl = f"{int(best_short*60)} min" if best_short < 1 else f"{best_short:.0f} h"
print(f"\npaired per-subject, {lbl} vs 24 h: mean diff {np.mean(a-b):+.3f}, "
      f"Wilcoxon p = {pv:.4f}")
print()
if max(f1_short) - min(f1_long) < 0.03 and pv > 0.05:
    print("OUTCOME: sub-recording periods perform like supra-recording ones. The")
    print("invariance is NOT an artefact of all periods collapsing to a linear")
    print("trend - it holds where the fitted shapes genuinely differ. This is a")
    print("materially stronger version of Section IV-D; replace the sweep table.")
else:
    print("OUTCOME: sub-recording periods behave differently from the long ones.")
    print("The original sweep's invariance was at least partly structural. Report")
    print("both regimes and revise the Section IV-D wording accordingly.")

GAP 2 RESULT: do sub-recording timescales behave differently?
sub-recording  (3 periods): F1 0.582 (range 0.576-0.587)
supra-recording(4 periods): F1 0.643 (range 0.639-0.647)
spread across ALL periods: 0.071

paired per-subject, 15 min vs 24 h: mean diff -0.059, Wilcoxon p = 0.0067

OUTCOME: sub-recording periods behave differently from the long ones.
The original sweep's invariance was at least partly structural. Report
both regimes and revise the Section IV-D wording accordingly.


## Gap 3 — dispersion for Table I

Two measures, because they answer different questions. Per-fold standard deviation
describes how much performance varies between subjects. The subject-level bootstrap
interval describes the uncertainty in the pooled estimate given only fifteen
subjects — resampling subjects rather than windows, since windows within a subject
are not independent.

In [12]:
def bootstrap_subject_ci(perfold, n_boot=10000, seed=42):
    rng = np.random.RandomState(seed)
    vals = np.array(list(perfold.values()), dtype=float)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

X5, y5, g5 = build_at_step(5)
print("="*84)
print("GAP 3 RESULT: dispersion for Table I (step 5, session-mean level term)")
print("="*84)
print(f"{'configuration':<24}{'F1':>8}{'per-fold SD':>14}{'subject bootstrap 95% CI':>28}")
print("-"*84)
table1_disp = {}
for name, cols in CONFIGS.items():
    yt, yp, _, _, perfold = xgb_loso_predictions(X5, y5, g5, cols=cols)
    f1 = f1_score(yt, yp, average='macro', zero_division=0)
    vals = np.array(list(perfold.values()))
    lo, hi = bootstrap_subject_ci(perfold)
    table1_disp[name] = {'f1': float(f1), 'sd': float(vals.std(ddof=1)), 'ci': [lo, hi]}
    print(f"{name:<24}{f1:>8.3f}{vals.std(ddof=1):>14.3f}{f'[{lo:.3f}, {hi:.3f}]':>28}")
print("-"*84)

a = table1_disp['+ residual']; b = table1_disp['+ level + residual']
print(f"\n'+ residual' {a['f1']:.3f} vs '+ level + residual' {b['f1']:.3f}: "
      f"difference {a['f1']-b['f1']:+.3f}")
print(f"per-fold SD of each: {a['sd']:.3f}, {b['sd']:.3f}")
if abs(a['f1']-b['f1']) < min(a['sd'], b['sd']):
    print("The difference is smaller than the between-subject SD, supporting the")
    print("Section IV-B claim that it is fold variation rather than a real effect.")
else:
    print("The difference EXCEEDS the between-subject SD - the 'fold variation'")
    print("explanation in Section IV-B is not supported and should be reworded.")

json.dump(table1_disp, open(f'{SAVE}/gap3_dispersion.json','w'))
print("\nsaved gap3_dispersion.json")

GAP 3 RESULT: dispersion for Table I (step 5, session-mean level term)
configuration                 F1   per-fold SD    subject bootstrap 95% CI
------------------------------------------------------------------------------------
baseline (HRV+time)        0.551         0.148              [0.423, 0.568]
+ level                    0.602         0.162              [0.456, 0.616]
+ residual                 0.646         0.140              [0.534, 0.670]
+ level + residual         0.643         0.135              [0.530, 0.663]
------------------------------------------------------------------------------------

'+ residual' 0.646 vs '+ level + residual' 0.643: difference +0.003
per-fold SD of each: 0.140, 0.135
The difference is smaller than the between-subject SD, supporting the
Section IV-B claim that it is fold variation rather than a real effect.

saved gap3_dispersion.json


In [13]:
import numpy as np
_, _, _, _, pf_base = xgb_loso_predictions(X5, y5, g5, cols=CONFIGS['baseline (HRV+time)'])
_, _, _, _, pf_full = xgb_loso_predictions(X5, y5, g5, cols=CONFIGS['+ residual'])

subs = sorted(pf_base)
a = np.array([pf_full[s] for s in subs])
b = np.array([pf_base[s] for s in subs])

print(f"{'subj':>6}{'HRV-only':>12}{'+residual':>12}{'delta':>10}")
for s, x, y_ in zip(subs, b, a):
    print(f"{s:>6}{x:>12.3f}{y_:>12.3f}{y_-x:>+10.3f}")
print(f"\nN improved: {(a > b).sum()} of {len(a)}")
print(f"mean delta: {(a-b).mean():+.4f}")

  subj    HRV-only   +residual     delta
     2       0.486       0.460    -0.025
     3       0.410       0.673    +0.263
     4       0.238       0.503    +0.264
     5       0.333       0.347    +0.014
     6       0.461       0.515    +0.054
     7       0.600       0.667    +0.067
     8       0.543       0.670    +0.127
     9       0.245       0.364    +0.119
    10       0.624       0.734    +0.109
    11       0.619       0.705    +0.086
    13       0.534       0.695    +0.161
    14       0.597       0.737    +0.140
    15       0.401       0.518    +0.118
    16       0.620       0.721    +0.101
    17       0.743       0.756    +0.013

N improved: 14 of 15
mean delta: +0.1075


## What to write

**Gap 1.** If the pattern is stable, add a sentence to Section IV-B and delete the
overlap paragraph from Limitations:

> To confirm that the result is not an artefact of overlapping windows, the
> decomposition was repeated at step sizes of 20, 40 and 60 beats (83\%, 67\% and
> 50\% overlap). The residual contribution remains [X--Y] and continues to exceed
> the level term at every step.

**Gap 2.** If sub-recording periods behave the same, replace Table II rather than
appending to it — a sweep spanning 15 min to 24 h is a stronger result than one
spanning 3--24 h, and it pre-empts the objection instead of answering it later.

**Gap 3.** Add an SD column to Table I, and cite the bootstrap interval in Section
IV-B where the 0.646 to 0.642 drop is explained.

**If any outcome is negative**, that is worth more than the extra ten days: it is
better to find it now than to have a reviewer find it. Report it and scope the
claim accordingly.